# Structured Output pada Agen: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/04-keluaran-terstruktur-agen-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/04-keluaran-terstruktur-agen-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 4 - Solusi Latihan: Ekstraktor Laporan Insiden
Jalankan sel satu per satu dengan Shift+Enter

Mengubah catatan lapangan yang berantakan menjadi `LaporanInsiden` nested yang tervalidasi.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Literal, Optional

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.data import KlasifikasiInsiden

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    temperature=0,
)

## Langkah 2 - Skema laporan insiden
`LaporanInsiden` memuat daftar `Korban` dan klasifikasi `KlasifikasiInsiden`.

In [ ]:
class Korban(BaseModel):
    """Seseorang yang terlibat atau terdampak insiden."""
    peran: str = Field(description="Peran atau jabatan, contoh operator, kontraktor, welder")
    cedera: Literal["Tidak ada", "Pertolongan pertama", "Perawatan medis", "Hilang hari kerja"] = Field(
        description="Tingkat cedera yang dialami")


class LaporanInsiden(BaseModel):
    """Laporan insiden terstruktur dari catatan lapangan."""
    lokasi: str = Field(description="Lokasi kejadian, contoh GS-SR1 atau pad sumur KN-061")
    jam_kejadian: Optional[str] = Field(default=None, description="Jam kejadian format HH.MM, kosong bila tidak disebut")
    uraian: str = Field(max_length=200, description="Uraian singkat kejadian, maksimal 200 karakter")
    korban: list[Korban] = Field(description="Orang yang terdampak; daftar kosong bila tidak ada")
    tindakan_segera: list[str] = Field(description="Tindakan segera yang sudah dilakukan")
    klasifikasi: KlasifikasiInsiden = Field(description="Klasifikasi insiden (jenis, bahaya, tingkat keparahan, ringkasan)")


agent = create_agent(
    model,
    tools=[],
    system_prompt=(
        "Anda mengubah catatan lapangan menjadi laporan insiden HSE. Jangan menebak jam atau korban yang tidak disebut. "
        "Berikan jawaban akhir sebagai LaporanInsiden."
    ),
    response_format=LaporanInsiden,
)

## Langkah 3 - Catatan 1: baut jatuh tanpa korban

In [ ]:
teks = (
    "kmrn sore sekitar jam 4an di GS-SR1 ada baut jatuh dr pipe rack pas lifting. ga ada yg luka. "
    "area dibarikade, kerjaan distop dulu, spv udah dilapor."
)
print(f"Catatan : {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
laporan = response.get("structured_response")
if laporan is None:
    print("   Tidak ada hasil terstruktur\n")
else:
    print(laporan.model_dump_json(indent=2))
    print()

Catatan : kmrn sore sekitar jam 4an di GS-SR1 ada baut jatuh dr pipe rack pas lifting. ga ada yg luka. area dibarikade, kerjaan distop dulu, spv udah dilapor.
{
  "lokasi": "GS-SR1",
  "jam_kejadian": "16.00",
  "uraian": "Baut jatuh dari pipe rack saat lifting di GS-SR1. Tidak ada korban luka. Area dibarikade, pekerjaan dihentikan sementara, supervisor sudah dilaporkan.",
  "korban": [],
  "tindakan_segera": [
    "Area dibarikade",
    "Pekerjaan dihentikan sementara",
    "Supervisor sudah dilaporkan"
  ],
  "klasifikasi": {
    "incident_type": "Near Miss",
    "hazard": "Dropped Object",
    "severity": "Rendah",
    "ringkasan": "Baut jatuh dari pipe rack saat lifting di GS-SR1, tidak ada korban luka."
  }
}



## Langkah 4 - Catatan 2: hose pecah dengan korban

In [ ]:
teks = (
    "Pad sumur KN-061 pagi 09.40, hose hidrolik pecah, oli panas kena tangan operator (Pak Dedi). "
    "Dikasih P3K lalu dibawa ke klinik lapangan, dirawat dan boleh kembali bertugas. Area dibersihkan pakai absorbent."
)
print(f"Catatan : {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
laporan = response.get("structured_response")
if laporan is None:
    print("   Tidak ada hasil terstruktur\n")
else:
    print(laporan.model_dump_json(indent=2))
    print()

Catatan : Pad sumur KN-061 pagi 09.40, hose hidrolik pecah, oli panas kena tangan operator (Pak Dedi). Dikasih P3K lalu dibawa ke klinik lapangan, dirawat dan boleh kembali bertugas. Area dibersihkan pakai absorbent.
{
  "lokasi": "Pad sumur KN-061",
  "jam_kejadian": "09.40",
  "uraian": "Hose hidrolik pecah dan oli panas mengenai tangan operator Pak Dedi. Korban diberikan P3K dan dibawa ke klinik lapangan, dirawat dan diperbolehkan kembali bertugas. Area dibersihkan dengan absorbent.",
  "korban": [
    {
      "peran": "operator",
      "cedera": "Pertolongan pertama"
    }
  ],
  "tindakan_segera": [
    "Pemberian P3K",
    "Evakuasi ke klinik lapangan",
    "Pembersihan area dengan absorbent"
  ],
  "klasifikasi": {
    "incident_type": "First Aid Case",
    "hazard": "Spill/Release",
    "severity": "Rendah",
    "ringkasan": "Hose hidrolik pecah mengakibatkan operator terkena oli panas dan mendapatkan perawatan P3K."
  }
}



### Penjelasan Solusi Latihan 4.1: ekstraktor laporan insiden

Catatan lapangan yang berantakan ("kmrn sore jam 4an", "ga ada yg luka") diubah menjadi objek `LaporanInsiden` yang tervalidasi. Skemanya bersarang: laporan memuat daftar `Korban` dan satu `KlasifikasiInsiden`.

- `TODO(B4.1)` dan `TODO(B4.2)`: skema. `uraian` dibatasi 200 karakter, `cedera` dibatasi empat nilai, dan `jam_kejadian` boleh kosong.
- `TODO(B4.3)`: `response_format=LaporanInsiden` membuat agen menutup dengan objek bertipe. System prompt melarang menebak jam dan korban.
- `TODO(B4.4)`: hasilnya diambil dari `response["structured_response"]`. Pakai `.get(...)` dan periksa `None`, karena agen kadang gagal menghasilkan objek.

Dua hal pada keluaran layak dibaca teliti. Catatan pertama menyebut "jam 4an" dan model mengisi `16.00`; itu tafsiran, bukan fakta yang tertulis, jadi di aplikasi nyata jam seperti ini sebaiknya ditandai perlu dikonfirmasi. Catatan kedua menyebut korban "dirawat dan boleh kembali bertugas" dan model memilih `Pertolongan pertama`; kategori lain masuk akal, dan keputusan sebenarnya milik petugas HSE. Skema membatasi bentuk jawaban, bukan kebenarannya.

## Bab 4 - Solusi Latihan Bonus: Agen Triase Cerdas
Jalankan sel satu per satu dengan Shift+Enter

Menggabungkan tool pencari riwayat, Union skema (`WOExtract` atau `KlasifikasiInsiden`), dan keputusan eskalasi
yang diambil oleh kode Python biasa.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Union

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from migas_shared.config import DATA
from migas_shared.data import KlasifikasiInsiden, WOExtract

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    temperature=0,
)

## Langkah 2 - Tool riwayat work order dan agen triase
Tool menghitung riwayat work order sebuah aset dari data nyata. Agen memilih skema `WOExtract` atau
`KlasifikasiInsiden` sesuai isi pesan.

In [ ]:
_WO = pd.read_csv(DATA / "eval" / "work_orders_truth.csv")


@tool
def riwayat_work_order(asset_id: str) -> str:
    """Hitung berapa kali sebuah aset pernah masuk work order dan berapa total downtime-nya (jam).

    Args:
        asset_id: Kode aset, contoh SR-020 atau V-SR1-101
    """
    baris = _WO[_WO["asset_id"] == asset_id.strip().upper()]
    if baris.empty:
        return f"Tidak ada riwayat work order untuk {asset_id}."
    return (f"{asset_id}: {len(baris)} work order, total downtime {baris['downtime_hours'].sum():.1f} jam, "
            f"kategori kegagalan terbanyak {baris['failure_category'].mode().iat[0]}.")


agent = create_agent(
    model,
    tools=[riwayat_work_order],
    system_prompt=(
        "Anda memilah pesan lapangan. Untuk catatan perbaikan peralatan, panggil riwayat_work_order lebih dulu "
        "untuk aset yang disebut, lalu isi WOExtract. Untuk laporan kejadian keselamatan, isi KlasifikasiInsiden. "
        "Berikan jawaban akhir dalam skema yang sesuai."
    ),
    response_format=ToolStrategy(Union[WOExtract, KlasifikasiInsiden]),
)

## Langkah 3 - Keputusan eskalasi oleh kode
Keputusan eskalasi diambil oleh kode Python biasa, bukan oleh model.

In [ ]:
def keputusan(hasil) -> str:
    """Keputusan eskalasi oleh kode, bukan oleh model."""
    if isinstance(hasil, WOExtract):
        if hasil.safety_flag:
            return "ESKALASI: ada risiko keselamatan pada work order, beri tahu HSE"
        if hasil.downtime_hours >= 24:
            return "PERHATIAN: downtime 24 jam atau lebih, masukkan ke rapat produksi"
        return "RUTIN: jadwalkan sesuai prioritas pemeliharaan"
    if isinstance(hasil, KlasifikasiInsiden):
        return "ESKALASI: notifikasi manajer HSE" if hasil.severity == "Tinggi" else "RUTIN: catat di log HSE"
    return "TIDAK DIKENAL: periksa manual"

## Langkah 4 - Pesan 1: ESP trip, motor terbakar

In [ ]:
teks = "ESP SR-020 trip OL, coba restart 2x gagal. Megger motor 0 Mohm, motor burnt. Usul pulling ganti motor. downtime 5 jam."
print(f"Pesan: {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
hasil = response.get("structured_response")
print(f"   Skema terpilih : {type(hasil).__name__ if hasil else 'None'}")
print(f"   Isi            : {hasil}")
print(f"   Keputusan kode : {keputusan(hasil)}\n")

Pesan: ESP SR-020 trip OL, coba restart 2x gagal. Megger motor 0 Mohm, motor burnt. Usul pulling ganti motor. downtime 5 jam.
   Skema terpilih : WOExtract
   Isi            : asset_id='SR-020' equipment_type='ESP' failure_category='Electrical' component='Motor' action_category='Workover' downtime_hours=5.0 safety_flag=False
   Keputusan kode : RUTIN: jadwalkan sesuai prioritas pemeliharaan



## Langkah 5 - Pesan 2: gasket bocor dengan alarm LEL

In [ ]:
teks = "K-ML1-201 gasket flange bocor, ada alarm LEL menyala, diganti, downtime 6 jam."
print(f"Pesan: {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
hasil = response.get("structured_response")
print(f"   Skema terpilih : {type(hasil).__name__ if hasil else 'None'}")
print(f"   Isi            : {hasil}")
print(f"   Keputusan kode : {keputusan(hasil)}\n")

Pesan: K-ML1-201 gasket flange bocor, ada alarm LEL menyala, diganti, downtime 6 jam.
   Skema terpilih : WOExtract
   Isi            : asset_id='K-ML1-201' equipment_type='Gas Compressor' failure_category='Leak/Corrosion' component='Flange Gasket' action_category='Replace' downtime_hours=6.0 safety_flag=True
   Keputusan kode : ESKALASI: ada risiko keselamatan pada work order, beri tahu HSE



## Langkah 6 - Pesan 3: pekerja jatuh dari scaffold

In [ ]:
teks = "Pekerja kontraktor jatuh dari scaffold setinggi 3 meter di workshop, patah tulang lengan, tidak dapat bekerja beberapa minggu."
print(f"Pesan: {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
hasil = response.get("structured_response")
print(f"   Skema terpilih : {type(hasil).__name__ if hasil else 'None'}")
print(f"   Isi            : {hasil}")
print(f"   Keputusan kode : {keputusan(hasil)}\n")

Pesan: Pekerja kontraktor jatuh dari scaffold setinggi 3 meter di workshop, patah tulang lengan, tidak dapat bekerja beberapa minggu.
   Skema terpilih : KlasifikasiInsiden
   Isi            : incident_type='Lost Time Injury' hazard='Working at Height' severity='Tinggi' ringkasan='Pekerja kontraktor jatuh dari scaffold setinggi 3 meter di workshop, alami patah tulang lengan dan tidak dapat bekerja.'
   Keputusan kode : ESKALASI: notifikasi manajer HSE



### Penjelasan Solusi Latihan 4.2: triase dengan Union

Satu agen menerima dua jenis pesan, catatan perbaikan (`WOExtract`) dan laporan keselamatan (`KlasifikasiInsiden`), lalu memilih skema yang cocok.

- `TODO(B4.5)`: tool `riwayat_work_order` menghitung riwayat aset dari data CSV. Aset yang tidak punya riwayat dijawab dengan pesan, bukan error.
- `TODO(B4.6)`: `ToolStrategy(Union[WOExtract, KlasifikasiInsiden])` menyerahkan pilihan skema kepada model.
- `TODO(B4.7)`: fungsi `keputusan` mengambil keputusan eskalasi dengan `if` biasa. Ini inti latihannya: model mengekstrak, kode memutuskan.

Hasilnya sesuai harapan pada keluaran: pesan ESP SR-020 menjadi `WOExtract` dengan `safety_flag=False` dan diputuskan RUTIN; pesan gasket bocor dengan alarm LEL menjadi `WOExtract` dengan `safety_flag=True` dan dieskalasi ke HSE; laporan pekerja jatuh dari scaffold menjadi `KlasifikasiInsiden` bertingkat Tinggi dan dieskalasi. Karena aturan eskalasi ada di kode, Anda bisa mengujinya tanpa memanggil model, dan hasilnya selalu sama untuk masukan yang sama.